# Lesson 7.2 — Redundancy Resolution
**Module 6 · Unit 7 · Lesson 26**

Redundant arm: q̇ = J⁺ξ + (I − J⁺J)z. The pseudoinverse is minimum-norm; the null-space term is self-motion (no tool effect).

In [1]:
import numpy as np
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i]); M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def Jv_planar(P,T,q): return geometric_jacobian(P,T,q)[:2,:]
def fk_xy(P,T,q):
    M=forward_chain(P,T,q)[-1]; return M[:2,3].copy()
def dls_inv(J,lam): return J.T@np.linalg.inv(J@J.T+lam**2*np.eye(J.shape[0]))
P2=[(0,0,1,0),(0,0,1,0)]; T2=["R","R"]
P3=[(0,0,1,0),(0,0,1,0),(0,0,0.6,0)]; T3=["R","R","R"]


## Pseudoinverse achieves the task with minimum joint effort

In [2]:
checks=[]
Jr=Jv_planar(P3,T3,np.array([0.3,0.5,-0.4])); xi=np.array([0.4,0.1])
Jp=np.linalg.pinv(Jr); qd_min=Jp@xi
print("J+ xi =",np.round(qd_min,3),"  J q_dot =",np.round(Jr@qd_min,4),"(== xi)")
checks.append(np.allclose(Jr@qd_min,xi))

J+ xi = [ 0.741 -1.597  0.836]   J q_dot = [0.4 0.1] (== xi)


## Null-space term is self-motion (no tool twist); pinv is minimum-norm

In [3]:
N=np.eye(3)-Jp@Jr; z=np.array([1.0,-2.0,0.5]); qd_null=N@z
print("J (null term) =",np.round(Jr@qd_null,8),"(== 0)")
checks.append(np.allclose(Jr@qd_null,0,atol=1e-9))
qd_alt=qd_min+qd_null
print("||q_dot|| pinv =",round(np.linalg.norm(qd_min),3)," with null term =",round(np.linalg.norm(qd_alt),3),"(>= pinv)")
checks.append(np.linalg.norm(qd_alt)>=np.linalg.norm(qd_min)-1e-12)
checks.append(np.allclose(Jr@qd_alt,xi))   # secondary task preserves tool twist

J (null term) = [-0.  0.] (== 0)
||q_dot|| pinv = 1.949  with null term = 2.014 (>= pinv)


## Secondary objective via null space (e.g. drive joint 2 toward center)

In [4]:
q=np.array([0.3,0.5,-0.4]); q_center=np.zeros(3)
z=(q_center-q)            # 'stay centered' secondary velocity
qd=Jp@xi + N@z
print("tool twist with secondary objective:",np.round(Jr@qd,4),"(still == xi)")
checks.append(np.allclose(Jr@qd,xi))
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

tool twist with secondary objective: [0.4 0.1] (still == xi)
All checks passed.
